In [2]:
import os
import pandas as pd
import numpy as np
import random
import pickle
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"


In [3]:
print(f"Pandas version: {pd.__version__},\nNumpy Version: {np.__version__}")

Pandas version: 2.2.3,
Numpy Version: 2.2.6


In [4]:
pd.set_option('display.max_columns', None)

### Pre-process functions

In [5]:
%run 00_functions_preprocess.ipynb

# Reading data

In [14]:
tokens_timeout_home_team = ['p550_1(1a)p200_9(0)', 'p500_1(3)p200_9(0)', 'p550_1(3a)p200_9(0)',
                            'p550_1(3a)p200_9(0)p440_8(0)', 'p550_1(5a)p200_9(0)', 'p200_9(0)',
                            'p200_9(0)p001_18(0)', 'p200_9(0)p440_8(0)', 'p200_9(0)p550_8(0)',
                            'p200_9(0)p440_8(0)p440_8(0)', 'p200_9(0)p440_8(0)p550_8(0)',
                            'p200_9(0)p550_8(0)p550_8(0)', 'p200_9(0)p440_8(0)p440_8(0)p440_8(0)',
                            'p200_9(0)p440_8(0)p440_8(0)p550_8(0)', 'p200_9(0)p440_8(0)p550_8(0)p550_8(0)',
                            'p200_9(0)p440_8(0)p440_8(0)p440_8(0)p550_8(0)',
                            'p200_9(0)p440_8(0)p440_8(0)p550_8(0)p550_8(0)', 'p200_9(1)', 'p200_9(1)p001_18(0)']

tokens_timeout_away_team = ['p440_1(1a)p300_9(0)', 'p440_1(1a)p300_9(0)p550_8(0)', 'p400_1(3)p300_9(0)',
                            'p440_1(3a)p300_9(0)', 'p440_1(3a)p300_9(0)p550_8(0)', 'p440_1(5a)p300_9(0)',
                            'p300_9(0)', 'p300_9(0)p001_18(0)', 'p300_9(0)p440_8(0)', 'p300_9(0)p550_8(0)',
                            'p300_9(0)p440_8(0)p440_8(0)', 'p300_9(0)p440_8(0)p550_8(0)', 'p300_9(0)p550_8(0)p550_8(0)',
                            'p300_9(0)p440_8(0)p440_8(0)p550_8(0)', 'p300_9(0)p440_8(0)p550_8(0)p550_8(0)',
                            'p300_9(0)p550_8(0)p550_8(0)p550_8(0)', 'p300_9(0)p440_8(0)p440_8(0)p550_8(0)p550_8(0)',
                            'p300_9(0)p440_8(0)p550_8(0)p550_8(0)p550_8(0)',
                            'p300_9(0)p440_8(0)p440_8(0)p550_8(0)p550_8(0)p550_8(0)', 'p300_9(1)', 'p300_9(1)p001_18(0)']

pbp_tokenized = pd.read_csv(data_folder + "pbp_tokenized.txt", sep = ";")\
.assign(numhtimeout = 7, numvtimeout = 7)\
.assign(numhtimeout = lambda df: np.where(df["period"] == 4, 4, df["numhtimeout"]))\
.assign(numvtimeout = lambda df: np.where(df["period"] == 4, 4, df["numvtimeout"]))\
.assign(numhtimeout = lambda df: np.where((df["period"] == 4) & (df["time"] <= 1800), 2, df["numhtimeout"]))\
.assign(numvtimeout = lambda df: np.where((df["period"] == 4) & (df["time"] <= 1800), 2, df["numvtimeout"]))\
.assign(numhtimeout = lambda df: np.where(df["period"] > 4, 2, df["numhtimeout"]))\
.assign(numvtimeout = lambda df: np.where(df["period"] > 4, 2, df["numvtimeout"]))\
.assign(timeout_home_call = lambda df: np.where(df["token"].isin(tokens_timeout_home_team), -1, 0))\
.assign(timeout_away_call = lambda df: np.where(df["token"].isin(tokens_timeout_away_team), -1, 0))\
.assign(tp_period = lambda df: np.where(df["period"] <= 4, "regular", df["period"]))\
.assign(available_hto = lambda df: df.groupby(["game_id", "tp_period"])["timeout_home_call"].cumsum())\
.assign(available_hto = lambda df: np.where(df["tp_period"] == "regular", df["available_hto"] + 7, df["available_hto"] + 2))\
.assign(available_hto = lambda df: np.where(df["available_hto"] < df["numhtimeout"], df["available_hto"], df["numhtimeout"]))\
.assign(available_ato = lambda df: df.groupby(["game_id", "tp_period"])["timeout_away_call"].cumsum())\
.assign(available_ato = lambda df: np.where(df["tp_period"] == "regular", df["available_ato"] + 7, df["available_ato"] + 2))\
.assign(available_ato = lambda df: np.where(df["available_ato"] < df["numvtimeout"], df["available_ato"], df["numvtimeout"]))\
.drop(columns = ["tp_period", "timeout_home_call", "timeout_away_call"])

tokens_info = pd.read_csv(data_folder + 'tokens_info.txt', sep = ';')\
.sort_values(['cod', 'token'])\
.set_index('token')

df_games_nba = pd.read_csv(data_folder + "games_nba.csv", sep = ';')\
[['GAME_ID', 'HOME_TEAM_ID', 'AWAY_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM', 'GAME_DATE', 'SEASON', "HOME_PTS", "AWAY_PTS"]]

pbp_tokenized.head()

,game_id,period,clock,token,last_time,time,spent_time,limhfouls,limvfouls,hpts,vpts,dif,numhtimeout,numvtimeout,available_hto,available_ato
0,21800001,1,12:00:00,p000_12(1)p455_10(0),7200,7200,0,4,4,0,0,0,7,7,7,7
1,21800001,1,11:40:00,p500_2(5)p200_4(0t),7200,7000,200,4,4,0,0,0,7,7,7,7
2,21800001,1,11:15:00,p400_2(5),7000,6750,250,4,4,0,0,0,7,7,7,7
3,21800001,1,11:13:00,p500_4(0p),6750,6730,20,4,4,0,0,0,7,7,7,7
4,21800001,1,11:08:00,p540_5(11p),6730,6680,50,4,4,0,0,0,7,7,7,7


In [15]:
pbp_tokenized = build_covariables(pbp_tokenized)
pbp_tokenized.head()

It's over


,game_id,period,clock,token,last_time,time,spent_time,limhfouls,limvfouls,hpts,vpts,dif,numhtimeout,numvtimeout,available_hto,available_ato,ahfbb,aafbb,ind_lastplay,period_,limhtimeout,limvtimeout,diff,dif_20,ind_close_3,ind_close_5,ind_close_8,ind_blowout_15,lead,home_lead,away_lead,ind_last_2_minutes,ind_last_minute,ind_last_20_seconds,ind_last_10_seconds,ind_last_5_seconds,ind_last_second,ind_last_2_minutes_game,ind_last_minute_game,ind_last_20_seconds_game,ind_last_10_seconds_game,ind_last_5_seconds_game,ind_clutch_2_last_minutes,ind_clutch_2_last_minutes_home,ind_clutch_2_last_minutes_away,ind_clutch_last_minute,ind_clutch_last_minute_home,ind_clutch_last_minute_away,ind_clutch_last_20_seconds,ind_clutch_last_20_seconds_home,ind_clutch_last_20_seconds_away,ind_clutch_last_10_seconds,ind_clutch_last_10_seconds_home,ind_clutch_last_10_seconds_away,ind_clutch_last_5_seconds,ind_clutch_last_5_seconds_home,ind_clutch_last_5_seconds_away,ind_blowout_last_minute,period1,period2,period3,period4,period_overtime,ball_possession,remaining_time
0,21800001,1,12:00:00,p000_12(1)p455_10(0),0.000000,0.000000,0,0,0,0,0,0.0,7,7,7,7,4,4,False,1,1,1,0,0.0,1,1,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,7200
1,21800001,1,11:40:00,p500_2(5)p200_4(0t),0.000000,0.027778,200,0,0,0,0,0.0,7,7,7,7,4,4,False,1,1,1,0,0.0,1,1,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,7000
2,21800001,1,11:15:00,p400_2(5),0.027778,0.062500,250,0,0,0,0,0.0,7,7,7,7,4,4,False,1,1,1,0,0.0,1,1,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,6750
3,21800001,1,11:13:00,p500_4(0p),0.062500,0.065278,20,0,0,0,0,0.0,7,7,7,7,4,4,False,1,1,1,0,0.0,1,1,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,6730
4,21800001,1,11:08:00,p540_5(11p),0.065278,0.072222,50,0,0,0,0,0.0,7,7,7,7,4,4,False,1,1,1,0,0.0,1,1,1,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.0,6680


In [16]:
pbp_tokenized["limhtimeout"].value_counts()

limhtimeout
1    2548853
0      16724
Name: count, dtype: int64

In [17]:
pbp_tokenized.describe(percentiles = [.01, .05, .1, .25, .5, .75, .90, .95, .99]).T.round(4).iloc[-60:]

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
last_time,2565577.0,0.5119,0.2938,0.0,0.0,0.0472,0.1014,0.2597,0.5153,0.7667,0.9174,0.9610,0.9981,1.0
time,2565577.0,0.5118,0.2939,0.0,0.0,0.0458,0.1014,0.2597,0.5153,0.7667,0.9174,0.9610,0.9981,1.0
spent_time,2565577.0,80.1965,68.3391,0.0,0.0,0.0000,0.0000,20.0000,60.0000,130.0000,180.0000,210.0000,250.0000,300.0
limhfouls,2565577.0,0.2549,0.4358,0.0,0.0,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0
limvfouls,2565577.0,0.2592,0.4382,0.0,0.0,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0
hpts,2565577.0,57.6311,34.1485,0.0,0.0,5.0000,11.0000,28.0000,57.0000,86.0000,104.0000,113.0000,126.0000,175.0
vpts,2565577.0,56.3858,33.5563,0.0,0.0,5.0000,11.0000,28.0000,56.0000,84.0000,102.0000,110.0000,124.0000,176.0
dif,2565577.0,0.0404,0.3674,-1.0,-0.9,-0.5667,-0.4000,-0.1667,0.0333,0.2667,0.5000,0.6667,1.0000,1.0
numhtimeout,2565577.0,6.0695,1.6487,2.0,2.0,2.0000,4.0000,4.0000,7.0000,7.0000,7.0000,7.0000,7.0000,7.0
numvtimeout,2565577.0,6.0695,1.6487,2.0,2.0,2.0000,4.0000,4.0000,7.0000,7.0000,7.0000,7.0000,7.0000,7.0


In [18]:
columns_boxscore = ["home_FGM", "away_FGM", "home_FGA", "away_FGA", "home_FGP", "away_FGP",
                    "home_3FGM", "away_3FGM", "home_3FGA", "away_3FGA", "home_3FGP", "away_3FGP",
                    "home_FTM", "away_FTM", "home_FTA", "away_FTA", "home_FTP", "away_FTP",
                    "home_RebOff", "away_RebOff", "home_RebDef", "away_RebDef", "home_RebTot",
                    "away_RebTot", "home_AST", "away_AST", "home_STL", "away_STL", "home_BLK",
                    "away_BLK", "home_TO", "away_TO", "home_FP", "away_FP", "home_points",
                    "away_points", "home_PMP", "away_PMP", "home_outcome", 'away_outcome']

df_boxscore = pd.read_csv(data_folder + 'boxscore_moving_average20.txt', sep = ';')\
.set_index("game_id")\
[columns_boxscore]

df_boxscore.describe(percentiles = [.01, .05, .1, .25, .5, .75, .90, .95, .99]).T.round(2)

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
home_FGM,7115.0,41.09,1.99,35.40,36.65,37.85,38.45,39.65,41.15,42.45,43.60,44.35,45.64,48.45
away_FGM,7115.0,41.14,2.02,35.30,36.70,37.85,38.45,39.70,41.15,42.55,43.70,44.40,45.74,48.65
home_FGA,7115.0,88.42,2.67,79.45,82.20,84.04,85.05,86.60,88.40,90.30,91.75,92.80,94.70,98.30
away_FGA,7115.0,88.44,2.68,79.70,82.20,84.00,85.10,86.60,88.40,90.35,91.80,92.80,94.69,98.05
home_FGP,7115.0,0.46,0.02,0.41,0.42,0.43,0.44,0.45,0.46,0.48,0.49,0.50,0.51,0.54
away_FGP,7115.0,0.47,0.02,0.41,0.42,0.43,0.44,0.45,0.47,0.48,0.49,0.50,0.51,0.54
home_3FGM,7115.0,12.22,1.77,7.50,8.71,9.50,10.00,10.95,12.10,13.35,14.60,15.45,16.70,18.15
away_3FGM,7115.0,12.23,1.78,7.30,8.70,9.45,9.95,11.00,12.10,13.35,14.55,15.40,16.80,18.00
home_3FGA,7115.0,34.05,4.20,21.60,24.70,27.45,29.10,31.25,33.70,36.75,39.70,41.61,44.05,50.55
away_3FGA,7115.0,34.02,4.20,21.60,24.60,27.34,29.05,31.20,33.70,36.70,39.55,41.50,44.19,50.55


In [19]:
# Convert the index of tokens_info to a list of tokens
list_tokens = tokens_info.index.to_list()

# Create a dictionary mapping each token to a unique integer, starting from 1
token_to_integer = {tok: idx + 1 for idx, tok in enumerate(list_tokens)}

# Assign a special integer (0) for the period (.)
token_to_integer['.'] = 0

# Create an inverse mapping from integers back to tokens
integer_to_token = {idx: tok for tok, idx in token_to_integer.items()}

tokens_info = tokens_info\
.query("hpts != 0 or vpts != 0 or limhteam != 0 or limvteam != 0")\
.to_dict(orient = 'index')


In [20]:
# Apply the function to assign splits
data_split = assign_data_split(pbp_tokenized[['game_id']].drop_duplicates())

In [21]:
df_features_teams = normalize_boxscore_variables(df_boxscore)\
.assign(playoff_game = lambda x: [1 if str(i)[0] != '2' else 0 for i in x.index])

df_features_teams["boxscore_list"]  = list(df_features_teams.values)

In [22]:
df_features_teams

,home_FGM,away_FGM,home_FGA,away_FGA,home_FGP,away_FGP,home_3FGM,away_3FGM,home_3FGA,away_3FGA,home_3FGP,away_3FGP,home_FTM,away_FTM,home_FTA,away_FTA,home_FTP,away_FTP,home_RebOff,away_RebOff,home_RebDef,away_RebDef,home_RebTot,away_RebTot,home_AST,away_AST,home_STL,away_STL,home_BLK,away_BLK,home_TO,away_TO,home_FP,away_FP,home_points,away_points,home_PMP,away_PMP,home_outcome,away_outcome,playoff_game,boxscore_list
game_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
21800001,0.100000,0.738889,0.169231,0.719231,0.202692,0.520951,0.250000,0.327778,0.276316,0.297368,0.388430,0.572594,0.475,0.460,0.510,0.575,0.455988,0.273684,0.233333,0.666667,0.330,0.845,0.270,1.045,0.125,0.940,0.36,0.81,0.2875,0.5000,0.128571,0.492857,0.516667,0.808333,0.075000,0.577083,0.5650,0.9800,0.60,0.75,0,"[0.09999999999999984, 0.7388888888888887, 0.16..."
21800002,0.566667,0.344444,0.326923,0.480769,0.628019,0.256217,0.361111,0.416667,0.360526,0.344737,0.532182,0.724247,0.385,0.675,0.230,0.845,0.866995,0.311279,0.183333,0.566667,0.560,0.340,0.470,0.480,0.575,0.050,0.59,0.90,0.7000,0.3750,0.314286,0.392857,0.466667,0.641667,0.429167,0.404167,0.9725,0.6150,0.75,0.55,0,"[0.5666666666666669, 0.3444444444444446, 0.326..."
21800003,0.438889,0.638889,0.323077,0.269231,0.482856,0.758934,0.261111,0.127778,0.155263,0.039474,0.703041,0.553398,0.975,0.265,1.105,0.215,0.554217,0.511166,0.241667,0.000000,0.635,0.205,0.580,0.005,0.215,0.490,0.25,0.75,0.3000,0.7375,0.328571,0.378571,-0.216667,0.700000,0.541667,0.345833,0.5525,0.4725,0.40,0.50,0,"[0.4388888888888892, 0.6388888888888888, 0.323..."
21800004,0.405556,0.350000,0.423077,0.334615,0.368254,0.370971,0.400000,0.722222,0.284211,0.678947,0.815789,0.825858,0.160,0.425,0.175,0.425,0.261603,0.501873,0.300000,0.116667,0.575,0.385,0.555,0.255,0.300,0.600,0.57,0.36,0.4125,0.8625,0.292857,0.535714,0.116667,0.516667,0.229167,0.418750,0.5925,0.3700,0.50,0.40,0,"[0.4055555555555554, 0.34999999999999987, 0.42..."
21800005,0.450000,0.266667,0.280769,0.261538,0.528897,0.329430,0.138889,0.166667,0.028947,0.052632,0.620352,0.653846,0.055,0.125,-0.040,0.105,0.465909,0.320210,0.300000,0.325000,0.185,0.005,0.165,0.000,0.250,0.240,0.74,0.51,0.1875,0.3375,0.242857,0.650000,0.425000,0.791667,0.120833,0.022917,0.6225,-0.0500,0.55,0.20,0,"[0.4499999999999997, 0.2666666666666665, 0.280..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22300813,1.027778,0.694444,0.492308,0.684615,1.020865,0.497739,0.505556,0.416667,0.350000,0.528947,0.965245,0.352354,0.710,0.300,0.785,0.260,0.517086,0.511327,-0.050000,0.491667,0.360,0.260,0.130,0.355,1.075,1.015,0.57,0.53,0.7375,0.8500,0.335714,0.535714,-0.241667,-0.183333,0.964583,0.510417,0.5425,0.2425,0.60,0.25,0,"[1.0277777777777777, 0.6944444444444444, 0.492..."
22300814,0.833333,0.444444,0.657692,0.153846,0.671084,0.624339,0.488889,0.338889,0.363158,0.331579,0.887147,0.530351,0.315,0.400,0.355,0.490,0.329466,0.282387,0.466667,0.483333,0.320,-0.020,0.440,0.095,0.775,0.480,0.17,0.59,0.3125,0.4125,0.507143,0.428571,0.550000,0.533333,0.647917,0.335417,0.1900,0.4625,0.25,0.50,0,"[0.8333333333333334, 0.4444444444444444, 0.657..."
22300815,0.688889,0.522222,0.234615,0.465385,0.846430,0.469304,0.605556,0.655556,0.397368,0.686842,1.132104,0.653088,0.460,0.235,0.490,0.185,0.457060,0.488665,0.200000,0.191667,0.455,0.090,0.410,0.005,0.715,0.740,0.46,0.54,0.6250,0.4625,0.407143,0.307143,0.458333,0.533333,0.643750,0.443750,0.9700,0.2800,0.65,0.30,0,"[0.6888888888888892, 0.5222222222222226, 0.234..."


In [23]:
df_features_teams.describe(percentiles = [.01, .05, .1, .25, .5, .75, .90, .95, .99]).T.round(2)

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
home_FGM,7115.0,0.57,0.22,-0.07,0.07,0.21,0.27,0.41,0.57,0.72,0.84,0.93,1.07,1.38
away_FGM,7115.0,0.57,0.22,-0.08,0.08,0.21,0.27,0.41,0.57,0.73,0.86,0.93,1.08,1.41
home_FGA,7115.0,0.49,0.21,-0.20,0.02,0.16,0.23,0.35,0.49,0.64,0.75,0.83,0.98,1.25
away_FGA,7115.0,0.50,0.21,-0.18,0.02,0.15,0.24,0.35,0.49,0.64,0.75,0.83,0.98,1.23
home_FGP,7115.0,0.50,0.21,-0.14,0.02,0.15,0.22,0.35,0.50,0.64,0.77,0.85,1.00,1.33
away_FGP,7115.0,0.50,0.21,-0.16,0.02,0.15,0.22,0.35,0.51,0.65,0.78,0.85,1.00,1.34
home_3FGM,7115.0,0.47,0.20,-0.06,0.08,0.17,0.22,0.33,0.46,0.59,0.73,0.83,0.97,1.13
away_3FGM,7115.0,0.47,0.20,-0.08,0.08,0.16,0.22,0.33,0.46,0.59,0.73,0.82,0.98,1.11
home_3FGA,7115.0,0.48,0.22,-0.18,-0.02,0.13,0.22,0.33,0.46,0.62,0.77,0.87,1.00,1.34
away_3FGA,7115.0,0.47,0.22,-0.18,-0.02,0.12,0.21,0.33,0.46,0.62,0.77,0.87,1.01,1.34


In [24]:
df_categorical = pbp_tokenized[["game_id"]]\
.drop_duplicates()\
.merge(df_features_teams, how = 'left', left_on = 'game_id', right_index = True)\
[['game_id', 'home_outcome', 'away_outcome']]\
.drop_duplicates()\
.pipe(put_categorical_training_outcomes)\
.rename(columns = {'ctgc': 'categorical_variable'})

In [25]:

df_categorical\
.merge(df_games_nba, how = 'left', left_on = "game_id", right_on = "GAME_ID")\
.assign(winner = lambda df: np.where(df["HOME_PTS"] > df["AWAY_PTS"], "home", "away"))\
.assign(n = lambda df: df.groupby("categorical_variable").transform("size"))\
.groupby(["categorical_variable", "n", "winner"], as_index = False).size()\
.assign(perc = lambda df: 100 * df["size"] / df["n"])\
.pivot_table(index = "categorical_variable", columns = "winner", values = "perc")

winner,away,home
categorical_variable,,
1,40.800000,59.200000
2,41.804320,58.195680
3,24.126456,75.873544
4,64.785374,35.214626
5,49.060150,50.939850


In [26]:

# 1: home past performance: average; away past performance: average
# 2: home past performance: good; away past performance: good
# 3: home past performance: good; away past performance: bad
# 4: home past performance: bad; away past performance: good
# 5: home past performance: bad; away past performance: bad

df_categorical['categorical_variable'].value_counts(normalize = True)


categorical_variable
1    0.281770
2    0.221753
4    0.177233
3    0.169343
5    0.149901
Name: proportion, dtype: float64

In [27]:
df_categorical_split = df_categorical\
.assign(randn = np.random.random(df_categorical.shape[0]))\
.assign(threshold_train = lambda df: df.groupby("categorical_variable")["randn"].transform(np.percentile, q = 80))\
.assign(threshold_dev = lambda df: df.groupby("categorical_variable")["randn"].transform(np.percentile, q = 90))\
.assign(categorical_split = lambda df: np.where(df["randn"] <= df["threshold_train"], "train", "other"))\
.assign(categorical_split = lambda df: np.where((df["randn"] <= df["threshold_dev"]) & (df["categorical_split"] == "other"), "dev", df["categorical_split"]))\
.assign(categorical_split = lambda df: np.where((df["categorical_split"] == "other"), "test", df["categorical_split"]))\
[["game_id", "categorical_variable", "categorical_split"]]

In [28]:
df_season_split = df_games_nba\
.assign(randn = np.random.random(df_games_nba.shape[0]))\
.assign(threshold = lambda df: df.groupby("SEASON")["randn"].transform(np.percentile, q = 90))\
.assign(season_split = lambda df: np.where(df["randn"] <= df["threshold"], "train", "dev"))\
.rename(columns = {"GAME_ID": "game_id"})\
.query("SEASON >= '2018-19'")\
[["game_id", "season_split"]]

In [29]:
df_token_features = pbp_tokenized\
.merge(data_split, on = 'game_id')\
.merge(df_categorical, on = 'game_id')\
.drop(columns = ["home_outcome", "away_outcome"])\
.merge(df_games_nba[["GAME_ID", "SEASON"]], how = "left", left_on = "game_id", right_on = "GAME_ID")\
.rename(columns = {"SEASON": "season"})\
.merge(df_season_split, how = "left")\
.merge(df_categorical_split, how = "left")\
.merge(df_features_teams, how = 'left', left_on = 'game_id', right_index = True)\
.pipe(func = build_token_tensors, token_context = 10, add_box_score_features = False)\
.assign(token_state = lambda df: categorize_time_bins(df["period"], df["remaining_time"], df["diff"]))\
[["game_id", "split", "season", "season_split", "categorical_variable", "categorical_split",
  "period", "remaining_time", "diff", "token_state", "ind_last_2_minutes_game", "limhfouls", 
  "limvfouls", "token_features", "numerical_features", "target_tokens"]]

In [31]:
df_token_features

,game_id,split,season,season_split,categorical_variable,categorical_split,period,remaining_time,diff,token_state,ind_last_2_minutes_game,limhfouls,limvfouls,token_features,numerical_features,target_tokens
0,21800001,train,2018-19,dev,2,train,1,7200.0,0.0,4,0.0,0.0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",87
1,21800001,train,2018-19,dev,2,train,1,7200.0,0.0,4,0.0,0.0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 87]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",152
2,21800001,train,2018-19,dev,2,train,1,7000.0,0.0,4,0.0,0.0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 87, 152]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.02777777777777779,...",144
3,21800001,train,2018-19,dev,2,train,1,6750.0,0.0,4,0.0,0.0,0.0,"[0, 0, 0, 0, 0, 0, 0, 87, 152, 144]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0625, 0.0, 0.0, 1....",240
4,21800001,train,2018-19,dev,2,train,1,6730.0,0.0,4,0.0,0.0,0.0,"[0, 0, 0, 0, 0, 0, 87, 152, 144, 240]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.06527777777777777,...",281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2565572,52200211,train,2022-23,train,1,train,4,650.0,27.0,161,1.0,0.0,1.0,"[542, 63, 28, 13, 12, 145, 239, 322, 334, 449]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9097222222222222, ...",334
2565573,52200211,train,2022-23,train,1,train,4,508.0,27.0,170,0.0,0.0,1.0,"[63, 28, 13, 12, 145, 239, 322, 334, 449, 334]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9294444444444444, ...",282
2565574,52200211,train,2022-23,train,1,train,4,475.0,27.0,170,0.0,0.0,1.0,"[28, 13, 12, 145, 239, 322, 334, 449, 334, 282]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9340277777777778, ...",29
2565575,52200211,train,2022-23,train,1,train,4,377.0,25.0,170,0.0,0.0,1.0,"[13, 12, 145, 239, 322, 334, 449, 334, 282, 29]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9476388888888889, ...",337


In [32]:
df_token_features.to_parquet(data_folder + 'data_token_features.parquet', engine = 'pyarrow', index = False)

In [ ]:
# with open(data_folder + 'data_token_features.pkl', 'wb') as f:
#     pickle.dump(df_token_features, f)

In [33]:
df_time_features = pbp_tokenized\
.merge(data_split, on = 'game_id')\
.merge(df_categorical, on = 'game_id')\
.drop(columns = ["home_outcome", "away_outcome"])\
.merge(df_games_nba[["GAME_ID", "SEASON"]], how = "left", left_on = "game_id", right_on = "GAME_ID")\
.rename(columns = {"SEASON": "season"})\
.merge(df_season_split, how = "left")\
.merge(df_categorical_split, how = "left")\
.merge(df_features_teams, how = 'left', left_on = 'game_id', right_index = True)\
.pipe(func = build_time_tensors, token_context = 10, add_box_score_features = False)\
.assign(token_state = lambda df: categorize_time_bins(df["period"], df["remaining_time"], df["diff"]))\
[["game_id", "split", "season", "season_split", "categorical_variable", "categorical_split",
  "period", "remaining_time", "diff", "token_state", "ind_last_2_minutes_game", "limhfouls", 
  "limvfouls", "token_features", "numerical_features", "target_time"]]

In [34]:
df_time_features

,game_id,split,season,season_split,categorical_variable,categorical_split,period,remaining_time,diff,token_state,ind_last_2_minutes_game,limhfouls,limvfouls,token_features,numerical_features,target_time
0,21800001,train,2018-19,dev,2,train,1,7200.0,0,4,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 87]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",0
1,21800001,train,2018-19,dev,2,train,1,7200.0,0,4,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 0, 87, 152]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",200
2,21800001,train,2018-19,dev,2,train,1,7000.0,0,4,0,0,0,"[0, 0, 0, 0, 0, 0, 0, 87, 152, 144]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.02777777777777779,...",250
3,21800001,train,2018-19,dev,2,train,1,6750.0,0,4,0,0,0,"[0, 0, 0, 0, 0, 0, 87, 152, 144, 240]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0625, 0.0, 0.0, 1....",20
4,21800001,train,2018-19,dev,2,train,1,6730.0,0,4,0,0,0,"[0, 0, 0, 0, 0, 87, 152, 144, 240, 281]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.06527777777777777,...",50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2565572,52200211,train,2022-23,train,1,train,4,650.0,27,161,0,0,1,"[63, 28, 13, 12, 145, 239, 322, 334, 449, 334]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9097222222222222, ...",142
2565573,52200211,train,2022-23,train,1,train,4,508.0,27,170,0,0,1,"[28, 13, 12, 145, 239, 322, 334, 449, 334, 282]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9294444444444444, ...",33
2565574,52200211,train,2022-23,train,1,train,4,475.0,25,170,0,0,1,"[13, 12, 145, 239, 322, 334, 449, 334, 282, 29]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9340277777777778, ...",98
2565575,52200211,train,2022-23,train,1,train,4,377.0,25,170,0,0,1,"[12, 145, 239, 322, 334, 449, 334, 282, 29, 337]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9476388888888889, ...",241


In [35]:
df_time_features.to_parquet(data_folder + 'data_time_features.parquet', engine = 'pyarrow', index = False)

In [ ]:
# with open(data_folder + 'data_time_features.pkl', 'wb') as f:
#     pickle.dump(df_time_features, f)

In [52]:
with open(data_folder + 'integer_to_token.pkl', 'wb') as f:
    pickle.dump(integer_to_token, f)

In [36]:
df_categorical.groupby("categorical_variable")[["home_outcome", "away_outcome"]].describe()

# 1: home past performance: average; away past performance: average
# 2: home past performance: good; away past performance: good
# 3: home past performance: good; away past performance: bad
# 4: home past performance: bad; away past performance: good
# 5: home past performance: bad; away past performance: bad

home_outcome                                             \
                            count      mean       std  min   25%   50%   75%   
categorical_variable                                                           
1                          2000.0  0.502975  0.067210  0.4  0.45  0.50  0.55   
2                          1574.0  0.649619  0.093017  0.5  0.60  0.65  0.70   
3                          1202.0  0.637562  0.096791  0.5  0.55  0.65  0.70   
4                          1258.0  0.308307  0.098115  0.0  0.25  0.30  0.40   
5                          1064.0  0.319173  0.096353  0.0  0.25  0.35  0.40   

                           away_outcome                                       \
                       max        count      mean       std  min   25%   50%   
categorical_variable                                                           
1                     0.60       2000.0  0.504100  0.066655  0.4  0.45  0.50   
2                     0.95       1574.0  0.655400  0.091429  0.5  0.60  0.65   
3                     0.95       1202.0  0.305740  0.095312  0.0  0.25  0.30   
4                     0.45       1258.0  0.641494  0.095873  0.5  0.55  0.65   
5                     0.45       1064.0  0.317528  0.099885  0.0  0.25  0.35   

                                  
                       75%   max  
categorical_variable              
1                     0.55  0.60  
2                     0.70  0.95  
3                     0.35  0.45  
4                     0.70  0.95  
5                     0.40  0.45

In [76]:
df_ = pd.read_parquet(data_folder + 'data_token_features.parquet',  engine = 'pyarrow')\
.assign(season = lambda df: df["season"].str.replace("\-", "", regex = True))

In [80]:
tokens_end_period = [91, 92, 93, 94, 95, 97, 98, 99, 154, 155, 156, 157, 158, 159]

# 2. Construindo o Grid Completo (o equivalente ao expand.grid do R)
unique_seasons = df_["season"].unique()
list_periods = [1, 2, 3, 4, 5]
list_remaining_times = np.arange(301)

full_grid = pd.MultiIndex.from_product(
    [unique_seasons, list_periods, list_remaining_times], 
    names = ["season", "period", "remaining_time"]
)

df_prop = df_\
.query("season_split == 'train' and target_tokens in @tokens_end_period")\
.assign(period = lambda df: np.minimum(df["period"], 5))\
.assign(remaining_time = lambda df: np.where(df["remaining_time"] > 300, 300, df["remaining_time"]))\
.groupby(["season", "period", "remaining_time"]).size()\
.reindex(full_grid, fill_value = 0)\
.reset_index(name = "size")\
.sort_values(["season", "period", "remaining_time"], ascending = [True, True, False])\
.assign(
    cumsum_size = lambda df: df.groupby(["season", "period"])["size"].cumsum(),
    total_size = lambda df: df.groupby(["season", "period"])["size"].transform("sum"),
    prop = lambda df: df["cumsum_size"] / df["total_size"]
)


dict_prop_end_period_remaining_time = {}

for season in df_prop['season'].unique():
    df_season = df_prop[df_prop['season'] == season]
    
    matriz_2d = df_season\
        .pivot(index = "period", columns = "remaining_time", values = "prop")\
        .fillna(0).values
    
    dict_prop_end_period_remaining_time[season] = matriz_2d

In [ ]:
with open(data_folder + 'prop_end_period_by_remaining_time.pkl', 'wb') as f:
    pickle.dump(dict_prop_end_period_remaining_time, f)